In [1]:
pip install tensorflow pydot

  Using cached absl_py-2.3.1-py3-none-any.whl.metadata (3.3 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached flatbuffers-25.9.23-py2.py3-none-any.whl.metadata (875 bytes)
  Using cached gast-0.6.0-py3-none-any.whl.metadata (1.3 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-py2.py3-none-win_amd64.whl.metadata (5.3 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached termcolor-3.2.0-py3-none-any.whl.metadata (6.4 kB)
  Using cached tensorboard-2.20.0-py3-none-any.whl.metadata (1.8 kB)
  Using cached keras-3.12.0-py3-none-any.whl.metadata (5.9 kB)
  Using cached markdown-3.10-py3-none-any.whl.metadata (5.1 kB)
  Using cached tensorboard_data_server-0.7.2-py3-none-any.whl.metadata (1.1 kB)
  Using cached rich-14.2.0-py3-none-any.whl.metadata (18 kB)
  Using cached namex-0.1.0-py3-none-any.whl.metadata (322 bytes)
  Using cached markdown_it_py-4.0.0-py3-none

In [3]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, RepeatVector, TimeDistributed
from tensorflow.keras.utils import plot_model
import os
import warnings

# Menonaktifkan peringatan TensorFlow yang tidak relevan
warnings.filterwarnings("ignore")
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# --- 1. DEFINISI KONFIGURASI (Sesuai Notebook 2 & 3 Anda) ---

# Dari Notebook 2, Cell 3
TIMESTEP = 2
# V1-V28 (28) + Amount (1) + Time_sin (1) + Time_cos (1)
N_FEATURES_FRAUD = 31  
# Sesuai 'final_olist_feature_cols' di Notebook 2, Cell 3
N_FEATURES_OLIST = 6   

# Dari Notebook 3, Cell 6
HIDDEN_DIM1 = 64
HIDDEN_DIM2 = 32
LATENT_DIM = 16

def create_lstm_ae_model(n_features, timestep, hidden_dim1, hidden_dim2, latent_dim, model_name):
    """
    Membuat arsitektur model LSTM-Autoencoder di Keras
    (Sesuai dengan arsitektur PyTorch Anda di Notebook 3, Cell 3)
    """
    
    # Input
    inputs = Input(shape=(timestep, n_features), name="Input_Sequence")

    # --- Encoder ---
    # Arsitektur ini meniru PyTorch: 
    # return_sequences=True lalu return_sequences=False untuk mendapatkan output terakhir
    encoder = LSTM(hidden_dim1, activation='relu', return_sequences=True, name="Encoder_LSTM_1")(inputs)
    encoder = LSTM(hidden_dim2, activation='relu', return_sequences=False, name="Encoder_LSTM_2")(encoder)
    
    # Bottleneck (vektor laten)
    latent_vector = Dense(latent_dim, activation='relu', name="Latent_Vector_Bottleneck")(encoder)

    # --- Decoder ---
    decoder = Dense(hidden_dim2, activation='relu', name="Decoder_Dense")(latent_vector)
    
    # Meniru PyTorch decoded.unsqueeze(1).repeat(1, self.timestep, 1)
    decoder = RepeatVector(timestep, name="Repeat_Vector")(decoder)
    
    decoder = LSTM(hidden_dim2, activation='relu', return_sequences=True, name="Decoder_LSTM_1")(decoder)
    decoder = LSTM(hidden_dim1, activation='relu', return_sequences=True, name="Decoder_LSTM_2")(decoder)
    
    # Output: aktivasi 'linear' (default) untuk merekonstruksi nilai asli
    output = TimeDistributed(Dense(n_features, activation='linear'), name="Output_Reconstructed")(decoder)

    # Gabungkan model
    model = Model(inputs=inputs, outputs=output, name=model_name)
    return model

print("Membuat arsitektur model...")

# --- 2. MEMBUAT MODEL UNTUK SETIAP DATASET ---
model_fraud = create_lstm_ae_model(
    N_FEATURES_FRAUD, TIMESTEP, 
    HIDDEN_DIM1, HIDDEN_DIM2, LATENT_DIM, 
    "Fraud_Kaggle_Model"
)
model_olist = create_lstm_ae_model(
    N_FEATURES_OLIST, TIMESTEP, 
    HIDDEN_DIM1, HIDDEN_DIM2, LATENT_DIM, 
    "Olist_Simulation_Model"
)

print("Model Fraud (Kaggle) Summary:")
model_fraud.summary()

print("\nModel Olist (Simulasi) Summary:")
model_olist.summary()

# --- 3. GENERATE GAMBAR DIAGRAM ---
output_path = "outputs/figures"
os.makedirs(output_path, exist_ok=True)

file_fraud = os.path.join(output_path, "FIG2_A_Fraud_Architecture.png")
file_olist = os.path.join(output_path, "FIG2_B_Olist_Architecture.png")

try:
    # Membuat diagram untuk model Fraud (Kaggle)
    plot_model(
        model_fraud,
        to_file=file_fraud,
        show_shapes=True,           # Tampilkan bentuk (shape) data
        show_layer_names=True,      # Tampilkan nama layer
        show_layer_activations=True, # Tampilkan fungsi aktivasi
        rankdir='LR'                # Atur layout Kiri-ke-Kanan (Left-to-Right)
    )
    print(f"\nBERHASIL: Diagram arsitektur model Fraud disimpan di:\n{file_fraud}")

    # Membuat diagram untuk model Olist (Simulasi)
    plot_model(
        model_olist,
        to_file=file_olist,
        show_shapes=True,
        show_layer_names=True,
        show_layer_activations=True,
        rankdir='LR'
    )
    print(f"\nBERHASIL: Diagram arsitektur model Olist disimpan di:\n{file_olist}")

    print("\nSelesai! Anda sekarang dapat menggunakan file .png tersebut di jurnal Anda.")

except ImportError:
    print("\n--- ERROR ---")
    print("Gagal men-generate diagram. 'plot_model' membutuhkan 'pydot' DAN 'Graphviz'.")
    print("Anda sudah meng-install 'pydot' (pip install pydot).")
    print("TAPI, Anda mungkin juga perlu meng-install 'Graphviz' di sistem Anda.")
    print("Silakan download Graphviz dari: https://graphviz.org/download/")
    print("Setelah install Graphviz, restart terminal Anda dan jalankan script ini lagi.")
except Exception as e:
    print(f"\n--- ERROR LAIN TERJADI ---")
    print(f"Detail: {e}")
    print("Pastikan TensorFlow dan pydot ter-install dengan benar.")

Membuat arsitektur model...
Model Fraud (Kaggle) Summary:


Model: "Fraud_Kaggle_Model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ Input_Sequence (InputLayer)          │ (None, 2, 31)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ Encoder_LSTM_1 (LSTM)                │ (None, 2, 64)               │          24,576 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ Encoder_LSTM_2 (LSTM)                │ (None, 32)                  │          12,416 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ Latent_Vector_Bottleneck (Dense)     │ (None, 16)                  │             528 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ Decoder_Dense (Dense)                │ (None, 32)                  │             544 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ Repeat_Vector (RepeatVector)         │ (None, 2, 32)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ Decoder_LSTM_1 (LSTM)                │ (None, 2, 32)               │           8,320 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ Decoder_LSTM_2 (LSTM)                │ (None, 2, 64)               │          24,832 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ Output_Reconstructed                 │ (None, 2, 31)               │           2,015 │
│ (TimeDistributed)                    │                             │                 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 73,231 (286.06 KB)

 Trainable params: 73,231 (286.06 KB)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ Input_Sequence (InputLayer)          │ (None, 2, 6)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ Encoder_LSTM_1 (LSTM)                │ (None, 2, 64)               │          18,176 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ Encoder_LSTM_2 (LSTM)                │ (None, 32)                  │          12,416 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ Latent_Vector_Bottleneck (Dense)     │ (None, 16)                  │             528 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ Decoder_Dense (Dense)                │ (None, 32)                  │             544 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ Repeat_Vector (RepeatVector)         │ (None, 2, 32)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ Decoder_LSTM_1 (LSTM)                │ (None, 2, 32)               │           8,320 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ Decoder_LSTM_2 (LSTM)                │ (None, 2, 64)               │          24,832 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ Output_Reconstructed                 │ (None, 2, 6)                │             390 │
│ (TimeDistributed)                    │                             │                 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 65,206 (254.71 KB)

 Trainable params: 65,206 (254.71 KB)

 Non-trainable params: 0 (0.00 B)

You must install graphviz (see instructions at https://graphviz.gitlab.io/download/) for `plot_model` to work.

BERHASIL: Diagram arsitektur model Fraud disimpan di:
outputs/figures\FIG2_A_Fraud_Architecture.png
You must install graphviz (see instructions at https://graphviz.gitlab.io/download/) for `plot_model` to work.

BERHASIL: Diagram arsitektur model Olist disimpan di:
outputs/figures\FIG2_B_Olist_Architecture.png

Selesai! Anda sekarang dapat menggunakan file .png tersebut di jurnal Anda.


In [4]:
python generate_diagram.py

SyntaxError: invalid syntax (3927374645.py, line 1)